# Forest Inspection - Evaluation

Danh gia model tren tap test (seq9). Chi can chay tuan tu tung cell.

In [ ]:
# === 0. SETUP ===
!pip install -q segmentation-models-pytorch albumentations

import os, re, time
import requests, zipfile
import numpy as np
import torch
import torch.nn as nn
from torch.amp import autocast
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

IS_KAGGLE = os.path.exists('/kaggle')
DATA_DIR   = Path('/kaggle/working/eval_data') if IS_KAGGLE else Path('data/forest_sunny')
OUTPUT_DIR = Path('/kaggle/working/outputs')   if IS_KAGGLE else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE} | Kaggle: {IS_KAGGLE}')

In [ ]:
# === 1. DOWNLOAD TEST DATA (seq9) ===
# Kaggle: chi can seq9 lam tap test
# Local : dat data vao data/forest_sunny/seq9/

TEST_SEQ = 'seq9'
ZENODO_BASE = 'https://zenodo.org/records/15511426/files'

def download_seq(seq, target_dir):
    seq_dir = target_dir / seq
    if seq_dir.exists() and list(seq_dir.rglob('*.png')):
        print(f'[OK] {seq} da co san ({len(list(seq_dir.rglob("*.png")))} anh)')
        return
    url = f'{ZENODO_BASE}/{seq}.zip?download=1'
    print(f'[DOWNLOAD] {seq} tu Zenodo...')
    r = requests.get(url, stream=True, timeout=120)
    total = int(r.headers.get('content-length', 0))
    zip_path = target_dir / f'{seq}.zip'
    target_dir.mkdir(parents=True, exist_ok=True)
    with open(zip_path, 'wb') as f:
        with tqdm(total=total, unit='B', unit_scale=True, desc=seq) as pbar:
            for chunk in r.iter_content(8192):
                if chunk: f.write(chunk); pbar.update(len(chunk))
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target_dir)
    zip_path.unlink()
    print(f'[OK] {seq} done')

if IS_KAGGLE:
    download_seq(TEST_SEQ, DATA_DIR)
else:
    print(f'[LOCAL] Dung data tai: {DATA_DIR / TEST_SEQ}')

In [ ]:
# === 2. CONSTANTS & DATASET ===

LABEL_COLORS = [
    (0,255,255),(0,127,0),(19,132,69),(0,53,65),(130,76,0),
    (152,251,152),(151,126,171),(250,150,0),(115,176,195),(123,123,123),(0,0,0),
]
CLASS_NAMES = [
    'Sky','Deciduous_trees','Coniferous_trees','Fallen_trees','Dirt_ground',
    'Ground_vegetation','Rocks','Building','Fence','Car','Empty',
]
NUM_CLASSES = len(CLASS_NAMES)

def rgb_to_mask(label_rgb):
    h, w, _ = label_rgb.shape
    out = np.full((h, w), NUM_CLASSES - 1, dtype=np.int64)
    for cid, c in enumerate(LABEL_COLORS):
        out[np.all(label_rgb == np.array(c, dtype=np.uint8), axis=-1)] = cid
    return out

def mask_to_rgb(mask):
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cid, c in enumerate(LABEL_COLORS):
        rgb[mask == cid] = c
    return rgb

class ForestDataset(Dataset):
    def __init__(self, seq_dir, transform=None):
        seq_dir = Path(seq_dir)
        self.transform = transform
        self.samples = [
            (cf, seq_dir / 'labels' / cf.name)
            for cf in sorted((seq_dir / 'color').glob('*.png'))
            if (seq_dir / 'labels' / cf.name).exists()
        ]
        print(f'Dataset: {len(self.samples)} samples')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        cp, lp = self.samples[idx]
        img = np.array(Image.open(cp).convert('RGB'))
        mask = rgb_to_mask(np.array(Image.open(lp).convert('RGB')))
        if self.transform:
            t = self.transform(image=img, mask=mask)
            img, mask = t['image'], t['mask']
        return img.float(), mask.long()

transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

# Tim thu muc seq (ho tro ca flat va nested: seq9/ hoac seq9/seq9/)
seq_dir = DATA_DIR / TEST_SEQ
if not (seq_dir / 'color').exists() and (seq_dir / TEST_SEQ / 'color').exists():
    seq_dir = seq_dir / TEST_SEQ

dataset = ForestDataset(seq_dir, transform=transform)
loader  = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=2)

In [ ]:
# === 3. LOAD MODEL ===
# Tim checkpoint tu:
#   - OUTPUT_DIR/checkpoints/  (local / cung session)
#   - /kaggle/input/*/         (output NB02 duoc add lam input)
#
# KAGGLE WORKFLOW:
#   1. Chay 02_train.ipynb -> Save Version (bat 'Save Output')
#   2. Mo notebook nay -> Add Data -> Your Work -> chon output cua 02_train

MODEL_NAME = 'unet'
ENCODER    = 'resnet34'

def parse_miou(p):
    m = re.search(r'miou_([0-9]+\.[0-9]+)', p.name)
    return float(m.group(1)) if m else 0.0

ckpts = list((OUTPUT_DIR / 'checkpoints').glob('*.pth')) if (OUTPUT_DIR / 'checkpoints').exists() else []
if IS_KAGGLE and Path('/kaggle/input').exists():
    ckpts += list(Path('/kaggle/input').rglob('*.pth'))

if ckpts:
    best = max(ckpts, key=parse_miou)
    print(f'Tim thay {len(ckpts)} checkpoint(s). Dung: {best.name}')
    CKPT_PATH = str(best)
else:
    CKPT_PATH = None
    print('[WARNING] Khong tim thay checkpoint!')
    if IS_KAGGLE:
        print('  -> Add Data -> Your Work -> chon output cua 02_train')

model = smp.Unet(encoder_name=ENCODER, encoder_weights=None, in_channels=3, classes=NUM_CLASSES)

if CKPT_PATH:
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    # Ho tro ca checkpoint tu Trainer (model_state_dict) va luu thang state_dict
    state = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state)
    print(f'[OK] Epoch={ckpt.get("epoch","?")}, mIoU={ckpt.get("metric",0):.4f}')
else:
    print('[WARNING] Dung random weights - ket qua se vo nghia.')

model = model.to(DEVICE).eval()

In [ ]:
# === 4. EVALUATE ===

cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
times = []

with torch.no_grad():
    for imgs, masks in tqdm(loader, desc='Evaluating'):
        imgs = imgs.to(DEVICE)
        t0 = time.time()
        with autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
            preds = model(imgs).argmax(dim=1).cpu().numpy().flatten()
        if DEVICE == 'cuda': torch.cuda.synchronize()
        times.append(time.time() - t0)
        tgts = masks.numpy().flatten()
        valid = (tgts >= 0) & (tgts < NUM_CLASSES)
        cm += np.bincount(tgts[valid] * NUM_CLASSES + preds[valid], minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)

inter = np.diag(cm)
union = cm.sum(1) + cm.sum(0) - inter
iou   = np.where(union > 0, inter / union, 0.0)
miou  = iou[union > 0].mean()
pacc  = inter.sum() / cm.sum()

print(f'\nmIoU:           {miou:.4f}')
print(f'Pixel Accuracy: {pacc:.4f}')
print(f'FPS:            {len(dataset)/sum(times):.1f}')
print('\nPer-class IoU:')
for name, v in zip(CLASS_NAMES, iou):
    print(f'  {name:22s} {v:.4f}  {"#"*int(v*30)}')

In [ ]:
# === 5. VISUALIZE PREDICTIONS (10 mau) ===

mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])
count = 0
out_dir = OUTPUT_DIR / 'predictions'
out_dir.mkdir(parents=True, exist_ok=True)

with torch.no_grad():
    for imgs, masks in loader:
        preds = model(imgs.to(DEVICE)).argmax(dim=1).cpu().numpy()
        for i in range(imgs.shape[0]):
            if count >= 10: break
            img_np = ((imgs[i].permute(1,2,0).numpy() * std + mean) * 255).clip(0,255).astype(np.uint8)
            fig, ax = plt.subplots(1, 3, figsize=(18, 5))
            ax[0].imshow(img_np);                ax[0].set_title('Input');       ax[0].axis('off')
            ax[1].imshow(mask_to_rgb(masks[i].numpy())); ax[1].set_title('Ground Truth'); ax[1].axis('off')
            ax[2].imshow(mask_to_rgb(preds[i]));         ax[2].set_title('Prediction');   ax[2].axis('off')
            patches = [mpatches.Patch(color=np.array(c)/255, label=n) for n, c in zip(CLASS_NAMES, LABEL_COLORS)]
            fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=8)
            plt.tight_layout()
            plt.savefig(out_dir / f'pred_{count:03d}.png', dpi=120, bbox_inches='tight')
            plt.show(); plt.close()
            count += 1
        if count >= 10: break

print(f'Saved {count} predictions -> {out_dir}')

In [ ]:
# === 6. CONFUSION MATRIX ===
try:
    import seaborn as sns
except ImportError:
    !pip install -q seaborn
    import seaborn as sns

cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Normalized Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()